# BTC-USD Regime Detection — Baseline

End-to-end walkthrough of the detection pipeline on BTC-USD hourly data.
This notebook is exploratory — it mirrors what `rde run --config configs/btc.yaml`
does, but exposes each step interactively.

> **Label disclaimer**: regime labels are heuristic interpretations of the model's
> state means and do **not** represent ground-truth market conditions.

In [ ]:
from pathlib import Path

from rde.config import load_config
from rde.data import YFinanceSource
from rde.features import FeaturePipeline, LogReturns, RollingVolatility, SmoothedReturns
from rde.models import select_n_states
from rde.inference import viterbi_decode, forward_backward_posteriors
from rde.evaluation import stationary_distribution, expected_dwell_times, empirical_dwell_times
from rde.labeling import rank_states
from rde.viz.interactive import (
    plot_price_with_regimes,
    plot_transition_heatmap,
    plot_regime_timeline,
    plot_per_regime_returns,
)

## 1. Load data

In [ ]:
cfg = load_config(Path('../configs/btc.yaml'))
source = YFinanceSource(cache_dir=Path('../results/cache'))
df = source.load(cfg.asset.symbol, cfg.asset.period, cfg.asset.interval)
print(f'{len(df)} rows  {df.index[0].date()} → {df.index[-1].date()}')
df.tail(3)

## 2. Feature engineering

In [ ]:
pipeline = FeaturePipeline([
    LogReturns(),
    RollingVolatility(window=24),
    SmoothedReturns(window=12),
])
df_feat = pipeline.transform(df)
print(f'Feature columns: {pipeline.output_columns}')
print(f'{len(df_feat)} rows after dropna')
df_feat[pipeline.output_columns].describe()

## 3. Model selection (AIC/BIC)

In [ ]:
import numpy as np

X = df_feat[pipeline.output_columns].values

fitted, scores_df = select_n_states(
    X,
    candidate_states=cfg.model.candidate_states,
    criterion=cfg.selection.criterion,
    n_restarts=cfg.model.n_restarts,
    covariance_type=cfg.model.covariance_type,
    n_iter=cfg.model.n_iter,
    seed_base=cfg.model.seed_base,
    feature_names=pipeline.output_columns,
)
print(f'Selected n_states={fitted.n_states}  BIC={fitted.bic:.2f}  log-lik={fitted.log_likelihood:.2f}')
scores_df

## 4. Decode regimes

In [ ]:
X_scaled = fitted.scaler.transform(X)
states    = viterbi_decode(fitted.hmm, X_scaled)
posteriors = forward_backward_posteriors(fitted.hmm, X_scaled)
print('Unique states:', np.unique(states))
print('Posterior shape:', posteriors.shape)

## 5. Heuristic labels

In [ ]:
labelled = rank_states(fitted)
label_list = [ls.label for ls in sorted(labelled, key=lambda x: x.index)]
for ls in labelled:
    count = int((states == ls.index).sum())
    print(f'  State {ls.index}: {ls.label!r:30s}  {count:6d} bars  ({100*count/len(states):.1f}%)')

## 6. Diagnostics

In [ ]:
import pandas as pd

stat = stationary_distribution(fitted.hmm.transmat_)
edt  = expected_dwell_times(fitted.hmm.transmat_)
emp  = empirical_dwell_times(states)

diag = pd.DataFrame({
    'label':          label_list,
    'stationary':     stat,
    'expected_dwell': edt,
    'empirical_mean': [emp.get(i, np.array([])).mean() if len(emp.get(i, np.array([]))) > 0 else float('nan') for i in range(fitted.n_states)],
})
diag

## 7. Visualisations (interactive)

In [ ]:
plot_price_with_regimes(df_feat, states, labels=label_list, symbol=cfg.asset.symbol).show()

In [ ]:
plot_transition_heatmap(fitted.hmm.transmat_, labels=label_list, symbol=cfg.asset.symbol).show()

In [ ]:
plot_regime_timeline(df_feat, states, labels=label_list, symbol=cfg.asset.symbol).show()

In [ ]:
plot_per_regime_returns(df_feat, states, labels=label_list, symbol=cfg.asset.symbol).show()